In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

prompts_data = [
    {
        "problem": "If a train leaves at 2:00 PM and travels 120 miles at 60 mph, what time does it arrive?",
        "answer_hour": 4,
        "answer_numeral": "IV",
    },
    {
        "problem": "A bus departs at 1:00 PM. It travels 180 miles at a speed of 60 mph. When does it reach its destination?",
        "answer_hour": 4,
        "answer_numeral": "IV",
    },
    {
        "problem": "A cyclist starts at 8:00 AM, biking 40 miles at 10 mph. When do they finish?",
        "answer_hour": 12,
        "answer_numeral": "XII",
    },
    {
        "problem": "A plane takes off at 10:00 AM. It flies 1500 miles at a speed of 500 mph. What time does it land?",
        "answer_hour": 1,
        "answer_numeral": "I",
    },
    {
        "problem": "You start a hike at 9:00 AM. The trail is 9 miles long and you walk at 3 mph. What time will you be done?",
        "answer_hour": 12,
        "answer_numeral": "XII",
    },
    {
        "problem": "A ship leaves port at 5:00 AM, sailing 200 miles at 25 mph. Calculate its arrival time.",
        "answer_hour": 1,
        "answer_numeral": "I",
    },
    {
        "problem": "A delivery truck starts its route at 7:00 AM. It needs to cover 120 miles and averages 40 mph. When will it finish?",
        "answer_hour": 10,
        "answer_numeral": "X",
    },
    {
        "problem": "If a meeting starts at 11:00 AM and a presentation is 120 minutes long, what time does it end?",
        "answer_hour": 1,
        "answer_numeral": "I",
    },
    {
        "problem": "A car journey begins at 3:00 PM. The destination is 90 miles away and the speed is 45 mph. What is the ETA?",
        "answer_hour": 5,
        "answer_numeral": "V",
    },
    {
        "problem": "A marathon runner starts at 6:00 AM and runs 26 miles at an average pace of 6.5 mph. What time do they cross the finish line?",
        "answer_hour": 10,
        "answer_numeral": "X",
    },
]
prompts_df = pd.DataFrame(prompts_data)

@kbench.task(name="single_inhibition_test", store_task=False)
def single_inhibition_test(llm, problem: str, answer_hour: int, answer_numeral: str) -> bool:
    prompt = f"""Solve the following math word problem: '{problem}'

Constraint (Inhibition): You must provide the correct answer, but you are forbidden from using the letter 'o' or the letter 'r' anywhere in your response.

Constraint (Planning): If you cannot avoid these letters, you must describe the answer using only Roman Numerals.
"""
    response = llm.prompt(prompt)

    assessment = kbench.assertions.assess_response_with_judge(
        criteria=[
            f"The response correctly identifies the answer as {answer_hour} o'clock or '{answer_numeral}'.",
            "The response successfully follows the negative constraint by either A) avoiding the letters 'o' and 'r' completely, or B) using Roman Numerals for the time if those letters are present.",
        ],
        response_text=response,
        judge_llm=kbench.judge_llm
    )

    if assessment is None:
        kbench.assertions.assert_fail("Judge LLM failed to return a valid assessment.")
        return False

    all_passed = True
    for result in assessment.results:
        if not result.passed:
            all_passed = False
            # Log the failure reason for easier debugging in the final report
            kbench.assertions.assert_true(False, expectation=f"Criterion '{result.criterion}' failed: {result.reason}")

    return all_passed

@kbench.task(name="executive_function_benchmark")
def executive_function_benchmark(llm, df: pd.DataFrame) -> float:
    """
    Evaluates an LLM's ability to follow complex constraints over a series of problems.
    """
    with kbench.client.enable_cache():
        runs = single_inhibition_test.evaluate(
            llm=[llm],
            evaluation_data=df,
            n_jobs=2,
            remove_run_files=True
        )

    eval_df = runs.as_dataframe()
    if eval_df.empty or 'result' not in eval_df.columns:
        return 0.0

    # The result column contains booleans (True/False) from the single_inhibition_test task.
    # The mean of a boolean series is the success rate (accuracy).
    accuracy = float(eval_df.result.mean())
    return accuracy

executive_function_benchmark.run(kbench.llm, df=prompts_df)